# Controlling LLM Generation

<a href="https://colab.research.google.com/github/HassanAlgoz/dl/blob/main/modules/Building_with_Deep_Learning/01-llms/04_llm_generate.ipynb" target="_blank">
  <img src="https://raw.githubusercontent.com/HassanAlgoz/dl/main/assets/Open%20in%20Colab-F9AB00.svg" alt="Open in Colab" height="50"/>
</a>

_Click the badge above to open and run this notebook in Google Colab!_

In [ ]:
# --- Setup: Clone repo & cd into correct folder (Colab only) ---
import os
import sys
import subprocess

if "google.colab" in sys.modules:
    repo_url = "https://github.com/HassanAlgoz/dl.git"
    lab_folder = "dl/modules/Building_with_Deep_Learning/01-llms"

    # Only clone if the folder doesn't exist
    if not os.path.exists(lab_folder):
        subprocess.run(["git", "clone", repo_url])

    # Change working directory to the lab folder
    os.chdir(lab_folder)


In [ ]:
from rich import print as pprint

## Set up

In [ ]:
!pip install -qqq torch
!pip install -Uqqq transformers datasets evaluate accelerate timm

### Suppress output logs

In [ ]:
import os
import logging

from huggingface_hub.utils import disable_progress_bars
from transformers.utils import logging as transformers_logging

# os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
# disable_progress_bars()
# transformers_logging.set_verbosity_error()
# transformers_logging.disable_progress_bar()
# logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

## Overview

### Open-weight models

Twelve models worth knowing in 2026, each with one standout strength.

1. [Llama 4 Scout](https://huggingface.co/meta-llama/): Meta's first natively multimodal open-weight model.

2. [DeepSeek V4](https://huggingface.co/deepseek-ai/): A Mixture-of-Experts model under MIT license with a native million-token context window. Near-frontier performance at a fraction of the cost per token.

3. [Qwen3](https://huggingface.co/qwen/): Alibaba's flagship open-weight model with switchable thinking and non-thinking modes, all under Apache 2.0.

4. [Gemma 4](https://huggingface.co/google/): Google's open-weight family released under Apache 2.0, with the widest language coverage of any model on this list.

5. [Phi 4](https://huggingface.co/microsoft/): Microsoft’s compact model trained almost entirely on synthetic, curated data. A practical choice for edge and on-device deployment.

6. [Mistral Small 3.1](https://huggingface.co/mistralai/): A VLM with a long context window that fits on a consumer laptop.

7. [Nemotron 3 Super](https://huggingface.co/nvidia/): NVIDIA’s hybrid MoE with a million-token context window. Fully open weights, datasets, and recipes, with strong results on agentic coding benchmarks.

8. [GLM 5.1](https://huggingface.co/THUDM/): The first open-weight model to top SWE-Bench Pro. Released under MIT with no commercial restrictions.

9. [Kimi K2.6](https://huggingface.co/baichuan-inc/): Competitive with leading closed models on coding while costing far less per million tokens. Available on Hugging Face under a Modified MIT license.

10. [StarCoder2](https://huggingface.co/bigcode/): One of the most transparent code models available.

11. [OLMo 2 (AI2)](https://huggingface.co/allenai/): The most complete example of open-source reproducibility on this list. Weights, training data, code, and full recipes all released under Apache 2.0.

12. [Falcon 3](https://huggingface.co/tiiuae/): A family of lightweight open-weight models built to run on a single GPU.

LLMs fall under the most generic task of: `text-generation`.

In [ ]:
import torch
from transformers import pipeline

checkpoint = "microsoft/Fara1.5-4B"
generator = pipeline(
    task="text-generation",
    model=checkpoint,
    device_map="auto",
    model_kwargs={
        "torch_dtype": torch.bfloat16
    }
)

Single input:

In [ ]:
output = generator(
    "What's the tallest mountain in the world?",
    max_new_tokens=30
  )

In [ ]:
print(output[0]['generated_text'])

Batch inference:

In [ ]:
output = generator([
    "Once upon a time,",
    "What if",
    "I want to go to",
    "When I was",
],
    max_new_tokens=30)
output

In [ ]:
print(output[0][0]['generated_text'])

In [ ]:
print(output[1][0]['generated_text'])

In [ ]:
print(output[2][0]['generated_text'])

In [ ]:
print(output[3][0]['generated_text'])

## Control Sequence Prediction

LLM [.generate()](https://huggingface.co/docs/transformers/v5.5.0/en/main_classes/text_generation#transformers.GenerationMixin.generate) method auto-regressively generates the next sequence given some initial `prompt`. Generation can be controlled:

See: [Common Options](https://huggingface.co/docs/transformers/v5.5.0/en/llm_tutorial#common-options):

- **`max_new_tokens` (`int`)**: Controls the maximum number of tokens generated. Make sure to set it explicitly—defaults are usually small.
- **`repetition_penalty` (`float`)**: Set to greater than 1.0 to discourage the model from repeating itself.
- **`temperature` (`float`)**: Influences randomness of generation. High values (`>0.8`) make outputs more creative; low values (`<0.4`) make them more focused and predictable.
- **`num_return_sequences` (`int`)**: Determines how many independent completions to generate for the given prompt.
- **`num_beams` (`int`)**: Activates beam search when set above 1. It is best suited for input-grounded tasks, like describing an image or speech recognition. See [this guide](https://huggingface.co/docs/transformers/v5.5.0/en/generation_strategies).

Difference between: `num_return_sequences` and `num_beams`: The latter  controls how many parallel paths the model evaluates internally, whereas the former, dictates the number of completely finished, independent text sequences handed back to you at the end of the process.

### Beams

Let's see the output of `num_beams=3`:

In [ ]:
# Note: __call__ and .generate() are the same
output = generator(
    "What is the color of the sky?",
    max_new_tokens=30,
    num_beams=3,
)

In [ ]:
print(output[0]['generated_text'])

### Multiple Output Sequences

In [ ]:
output = generator(
    "In this course, we will teach you how to",
    max_new_tokens=10,
    num_return_sequences=5, # each input produces 5 sequences
    temperature=1.5         # higher for more randomness / creative outputs
)

In [ ]:
for x in output:
    print(x['generated_text'])

Notice the difference?

## LLM for Specific Tasks

Instruction-tuned LLMs can complete sequences in respones to completing tasks.

Let's try to manually craft a prompt for text classification task.

In [ ]:
emotion_classifier_prompt = """
Analyze: "{text}"
Answer with one word "sad", "happy", or "depressed" and stop immediately.
"""

In [ ]:
prompt = emotion_classifier_prompt.format(
    text="I am feeling very joyful today!"
)
print(prompt)

 For close-ended tasks, we usually need to be more deterministic.

In [ ]:
output = generator(
    prompt,
    max_new_tokens=30,
    temperature=0.1,  # we want deterministic output
    # eos_token_id=generator.tokenizer.convert_tokens_to_ids("<output/>"),
)

In [ ]:
llm_output = output[0]['generated_text']
print(llm_output[len(prompt):])